# 03 Oxygen reduction (IA-ORR-001)

**Requirement:** `IA-ORR-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

$$
\newcommand{\per}{/}
\newcommand{\squared}{^{2}}
\newcommand{\cubed}{^{3}}
\newcommand{\of}[1]{\,\mathrm{#1}}
\newcommand{\milli}{\mathrm{m}}
\newcommand{\centi}{\mathrm{c}}
\newcommand{\micro}{\mathrm{\mu}}
\newcommand{\kilo}{\mathrm{k}}
\newcommand{\metre}{\mathrm{m}}
\newcommand{\gram}{\mathrm{g}}
\newcommand{\kilogram}{\mathrm{kg}}
\newcommand{\second}{\mathrm{s}}
\newcommand{\minute}{\mathrm{min}}
\newcommand{\hour}{\mathrm{h}}
\newcommand{\day}{\mathrm{d}}
\newcommand{\ampere}{\mathrm{A}}
\newcommand{\volt}{\mathrm{V}}
\newcommand{\watt}{\mathrm{W}}
\newcommand{\ohm}{\mathrm{\Omega}}
\newcommand{\kelvin}{\mathrm{K}}
\newcommand{\celsius}{{}^{\circ}\mathrm{C}}
\newcommand{\coulomb}{\mathrm{C}}
\newcommand{\joule}{\mathrm{J}}
\newcommand{\pascal}{\mathrm{Pa}}
\newcommand{\litre}{\mathrm{L}}
\newcommand{\mole}{\mathrm{mol}}
\newcommand{\molar}{\mathrm{M}}
\newcommand{\percent}{\%}
\newcommand{\atm}{\mathrm{atm}}
\newcommand{\newton}{\mathrm{N}}
\newcommand{\qty}[2]{#1\,#2}
\newcommand{\qtyrange}[3]{#1\text{--}#2\,#3}
\newcommand{\unit}[1]{#1}
\newcommand{\si}[1]{#1}
$$


## Governing equations

Oxygen reduction (US12308414B2):

$$\mathrm{O_2} + 2\,\mathrm{H_2O} + 4\,\mathrm{e^-} \to 4\,\mathrm{OH^-}$$

Modes: floating, vertical natural-air-breathing, inverse, tubular, stacked
submerged with depth-equalized pressure drop, wavy/rippled, bifunctional
(IA-SYS-009, IA-SYS-012; US Claims 1, 4, 7-12).
One face to electrolyte/channels, opposing face to air (EP Claim 1). PTFE/GDE parameterizable.
Current density in $\unit{\ampere\per\metre\squared}$; $p_{\mathrm{O_2}}$ in $\unit{\atm}$.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params
from plant_sim.siunitx_nb import SIUNITX_PREAMBLE

In [ ]:
from IPython import get_ipython
from IPython.display import Markdown, display

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
display(Markdown(SIUNITX_PREAMBLE))
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Isolated component simulation

In [ ]:
from plant_sim.components.orr import OxygenReduction, ORR_MODES, stacked_orr_flows
from plant_sim.components.base import integrate_component

print("ORR modes", ORR_MODES)
orr = OxygenReduction(P, mode="stacked_submerged")
t, y = integrate_component(
    lambda t, y: orr.rhs(
        t,
        y,
        {
            "I_orr_A": -20 * P.I_discharge_100h_A,
            "T_K": P.T_ep_sim_K,
            "c_O2_gdl_mol_m3": 8.0,
            "p_O2_Pa": 21200,
            "a_oh": 6.0,
            "a_h2o": 0.72,
        },
        {},
    ),
    orr.y0(),
    (0, 600),
    n_eval=120,
)
plot_frame = pd.DataFrame(
    {
        "t": (t),
        "c_tpb": (y[1]),
        "q_dl": (y[0]),
    }
)
plot_long = plot_frame.melt(id_vars="t", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("t (s)")
ax.set_ylabel("c(O₂) (mol m⁻³) / q (C)")
ax.set_title("ORR TPB oxygen")
ax.text(0.0, -0.22, "IA-ORR-001", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)
eq = stacked_orr_flows(P, 5e-4, True, P.T_ep_sim_K)
plot_frame = pd.DataFrame(
    {
        "x": (eq["z_m"]),
        "dP": (eq["dP_Pa"]),
    }
)
plot_long = plot_frame.melt(id_vars="x", var_name="quantity", value_name="value")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="x", y="value", hue="quantity", ax=ax, linewidth=2.0)
ax.set_xlabel("z (m)")
ax.set_ylabel("ΔP (Pa)")
ax.set_title("Stacked ORR ΔP (compensated)")
ax.text(0.0, -0.22, "US Claim 1", transform=ax.transAxes, fontsize=8, va="top")
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)

## Verification against patents / SSS002

In [ ]:
print("pO2 discharge window", P.pO2_discharge_min_atm, P.pO2_discharge_max_atm, "atm")